# 00b. Limpeza de dados

Entre o empilhamento (NB00) e a EDA (NB01). Duas classes de problema:

**Linha que não pode casar.** CPF com `ano_obito <= ANO_OBITO_CORTE` (hoje 2023)
sai da base. Registro sem `nome_completo` (nulo ou em branco) sai no Censo e no CPF.

**Valor-sentinela que o SQL lê como igualdade.** `''` e `'00000000'` são ausência
escrita como texto, mas `l.cep = r.cep` os trata como acordo real. Isso infla os
blocos e, nas `deterministic_rules` do NB02, contamina a estimativa do prior.
Viram `NULL`, que não casa com `NULL`.

Saída: `registro_limpo` e o parquet correspondente. O `materialize_splink_input`
passa a apontar para ela sozinho, então NB02 e NB03 não mudam.


In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

from IPython.display import display

PROB_DIR = Path.cwd()
if PROB_DIR.name == 'notebooks':
    PROB_DIR = PROB_DIR.parent
if str(PROB_DIR) not in sys.path:
    sys.path.insert(0, str(PROB_DIR))

import config
from config import (
    COHORT_DEDUP_ARQUIVO,
    COLUNAS_ESTRUTURAIS,
    REGISTRO_LIMPO,
    TABELA_LIMPA,
    sem_nome_sql,
    cpf_norm_sql,
    dob_valida_sql,
    cep_valido_sql,
    export_parquet,
    get_connection,
    limpeza_columns_sql,
    obito_antes_do_censo_sql,
    print_paths,
    require_tables,
)

ORIGEM = 'registro_unificado'

print_paths()
con = get_connection()
require_tables(con, [ORIGEM], notebook_origem='00')

tipos = {r[0]: r[1] for r in con.execute(f'DESCRIBE {ORIGEM}').fetchall()}
texto_cols = [
    c for c, t in tipos.items()
    if t.upper().startswith('VARCHAR') and c not in COLUNAS_ESTRUTURAIS
]
n_antes = con.execute(f'SELECT COUNT(*) FROM {ORIGEM}').fetchone()[0]
print(f'{ORIGEM}: {n_antes:,} linhas | {len(texto_cols)} colunas de texto a limpar')


OUTPUT_DIR: /home/ibge.gov.br/ramon.goncalves/data/probabilistico_output
CPF_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/cpf/cpf.parquet
CENSO_PESSOAS_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/censo/censo_pessoas_2022_20260505.parquet
CENSO_CEP_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/raw/censo/data_cep_uniq.csv
COHORT_DEDUP_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/capefe/dados/CohortDados/cohort_dedup.parquet
FILTRO_UF: None
FILTRO_MUNICIPIO: 2111300
USE_PHONETIC_STRIP_VOWELS: False
ANO_OBITO_CORTE: 2021
ANO_NASCIMENTO_MIN: 1900
SEXO_VALIDOS: ('M', 'F')
DUCKDB_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/data/probabilistico_output/probabilistico.duckdb
registro_unificado: 2,531,659 linhas | 21 colunas de texto a limpar


## 1. Diagnóstico antes

Quanto de cada coluna é string vazia hoje. São esses valores que o Splink
compara como se fossem iguais entre si.

In [2]:
vazios = ',\n    '.join(
    f"SUM(CASE WHEN TRIM(CAST({c} AS VARCHAR)) = '' THEN 1 ELSE 0 END) AS {c}"
    for c in texto_cols
)
df_vazios = con.execute(f'''
SELECT origem, COUNT(*) AS n_linhas,
    {vazios}
FROM {ORIGEM} GROUP BY origem ORDER BY origem
''').df().set_index('origem').T
display(df_vazios[df_vazios.sum(axis=1) > 0])

origem,censo,cpf
n_linhas,1037775.0,1493884.0
data_nascimento,171827.0,122.0


In [3]:
# Os valores mais frequentes denunciam preenchimento sintético. Se alguma data
# ou CEP aparecer com contagem fora de escala, é sentinela e não dado.
for col in ['data_nascimento', 'cep']:
    print(f'\n=== {col}: 15 valores mais frequentes ===')
    display(con.execute(f'''
    SELECT {col} AS valor, origem, COUNT(*) AS n
    FROM {ORIGEM}
    GROUP BY 1, 2 ORDER BY n DESC LIMIT 15
    ''').df())


=== data_nascimento: 15 valores mais frequentes ===


,valor,origem,n
0,,censo,171827
1,2004-09-09,cpf,196
2,2002-05-26,cpf,162
3,2004-08-26,cpf,160
4,2004-08-29,cpf,156
5,2005-01-11,cpf,154
6,2005-01-12,cpf,147
7,2005-01-10,cpf,131
8,2004-09-23,cpf,128
9,2004-07-15,cpf,127



=== cep: 15 valores mais frequentes ===


,valor,origem,n
0,65000000,cpf,303462
1,65010000,cpf,46573
2,65040000,cpf,16274
3,65046660,cpf,12816
4,65085000,cpf,11874
5,65035000,cpf,10712
6,65055000,cpf,9148
7,65058000,cpf,9129
8,65030000,cpf,8888
9,65045000,cpf,8738


In [4]:
# Distribuição completa de sexo: são poucos valores e cabe inteira. A coluna
# 'mantido' mostra o que sobrevive — o resto ('O' de outro, 'I' de ignorado,
# '9' de não informado) vira NULL. Se alguma categoria fora de M/F tiver volume
# relevante e for real, reveja SEXO_VALIDOS em config.py antes de seguir.
sexo_lista = ', '.join(f"'{v}'" for v in config.SEXO_VALIDOS)
display(con.execute(f'''
SELECT
    CASE WHEN TRIM(CAST(sexo AS VARCHAR)) = '' THEN '(vazio)' ELSE sexo END AS valor,
    origem,
    COUNT(*) AS n,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (PARTITION BY origem), 3) AS pct_origem,
    upper(TRIM(CAST(sexo AS VARCHAR))) IN ({sexo_lista}) AS mantido
FROM {ORIGEM}
GROUP BY 1, 2, 5 ORDER BY origem, n DESC
''').df())

,valor,origem,n,pct_origem,mantido
0,F,censo,554274,53.410,True
1,M,censo,483501,46.590,True
2,F,cpf,754379,50.498,True
3,M,cpf,739279,49.487,True
4,9,cpf,226,0.015,False


In [5]:
# Ano de óbito. 'sem óbito' inclui todo o Censo, que nunca tem a coluna.
display(con.execute(f'''
SELECT
    CASE
        WHEN ano_obito IS NULL THEN 'sem ano de óbito'
        WHEN ano_obito <= 0 THEN 'sentinela (<= 0)'
        WHEN ano_obito <= {config.ANO_OBITO_CORTE} THEN 'até o corte (sai)'
        ELSE 'depois do corte (fica)'
    END AS faixa,
    origem,
    COUNT(*) AS n
FROM {ORIGEM}
GROUP BY 1, 2 ORDER BY n DESC
''').df())

n_obito = con.execute(
    f'SELECT COUNT(*) FROM {ORIGEM} WHERE ano_obito IS NOT NULL'
).fetchone()[0]
if n_obito == 0:
    print(
        'ATENÇÃO: nenhum ano de óbito preenchido, e o NB00 falharia se '
        f'{config.CPF_COL_ANO_OBITO} faltasse no bronze. Então o mais provável é '
        'que este registro_unificado seja anterior à coluna: rode o NB00 de novo. '
        'O filtro de óbito abaixo não vai remover nada.'
    )
else:
    display(con.execute(f'''
    SELECT ano_obito, COUNT(*) AS n FROM {ORIGEM}
    WHERE ano_obito IS NOT NULL GROUP BY 1 ORDER BY ano_obito DESC LIMIT 30
    ''').df())


,faixa,origem,n
0,sem ano de óbito,cpf,1412195
1,sem ano de óbito,censo,1037775
2,anterior ao corte (sai),cpf,54941
3,posterior ao Censo,cpf,14316
4,entre o corte e o Censo,cpf,12432


,ano_obito,n
0,2025,3155
1,2024,5757
2,2023,5404
3,2022,5635
4,2021,6797
5,2020,6835
6,2019,4991
7,2018,4101
8,2017,3598
9,2016,2387


## 2. Aplicar a limpeza

Um passe só: o `WHERE` remove óbitos com `ano_obito <= ANO_OBITO_CORTE` e
registros sem nome (Censo e CPF); a projeção troca os sentinelas por `NULL`. A idade
do CPF acompanha a data de nascimento (anos completos em `DATA_REFERENCIA_IDADE`);
a do Censo vem de `PECP0401` e não é tocada.


In [6]:
cols = limpeza_columns_sql(tipos)
select_sql = ',\n    '.join(
    (alias if expr == alias else f'{expr} AS {alias}') for alias, expr in cols.items()
)
filtro_obito = obito_antes_do_censo_sql()
filtro_sem_nome = sem_nome_sql()
print('Filtro de óbito:', filtro_obito)
print('Filtro sem nome:', filtro_sem_nome)

n_rem_obito = con.execute(
    f'SELECT COUNT(*) FROM {ORIGEM} WHERE {filtro_obito}'
).fetchone()[0]
n_rem_nome = con.execute(f'''
SELECT COUNT(*) FROM {ORIGEM}
WHERE NOT {filtro_obito} AND {filtro_sem_nome}
''').fetchone()[0]

con.execute(f'''
CREATE OR REPLACE TABLE {TABELA_LIMPA} AS
SELECT
    {select_sql}
FROM {ORIGEM}
WHERE NOT {filtro_obito}
  AND NOT {filtro_sem_nome}
''')

n_depois = con.execute(f'SELECT COUNT(*) FROM {TABELA_LIMPA}').fetchone()[0]
print(
    f'{n_antes:,} → {n_depois:,} linhas '
    f'({n_rem_obito:,} por óbito, {n_rem_nome:,} sem nome)'
)

colunas_limpa = {r[0] for r in con.execute(f'DESCRIBE {TABELA_LIMPA}').fetchall()}
faltando = set(tipos) - colunas_limpa
if faltando:
    raise RuntimeError(f'Colunas perdidas na limpeza: {sorted(faltando)}')


Filtro de óbito: (TRY_CAST(ano_obito AS INTEGER) IS NOT NULL AND TRY_CAST(ano_obito AS INTEGER) > 0 AND TRY_CAST(ano_obito AS INTEGER) < 2021)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

2,531,659 → 2,476,718 linhas (54,941 removidas por óbito)


## 3. Diagnóstico depois

Duas coisas diferentes viram `NULL` e vale separá-las. **Reencodado** é o `''`
que já era ausência e só mudou de grafia. **Descartado** é valor que existia e
foi julgado inválido — data fora da faixa, CEP `00000000`. O segundo número é o
que merece revisão: se estiver alto, a regra pode estar agressiva demais.

In [7]:
import pandas as pd

linhas = []
for col in texto_cols:
    expr = cols[col]
    if expr == col:
        continue
    linhas.append(con.execute(f'''
    SELECT
        '{col}' AS coluna,
        SUM(CASE WHEN TRIM(CAST({col} AS VARCHAR)) = '' THEN 1 ELSE 0 END) AS reencodado,
        SUM(CASE WHEN TRIM(CAST({col} AS VARCHAR)) <> '' AND ({expr}) IS NULL
                 THEN 1 ELSE 0 END) AS descartado
    FROM {ORIGEM}
    WHERE NOT {filtro_obito} AND NOT {filtro_sem_nome}
    ''').df())

resumo = pd.concat(linhas, ignore_index=True)
resumo = resumo[(resumo['reencodado'] > 0) | (resumo['descartado'] > 0)]
display(resumo.sort_values('descartado', ascending=False))


,coluna,reencodado,descartado
16,data_nascimento,171948.0,34269.0
17,sexo,0.0,214.0


In [8]:
# Amostra do que foi descartado, para conferir se a regra faz sentido.
display(con.execute(f'''
SELECT data_nascimento, COUNT(*) AS n
FROM {ORIGEM}
WHERE TRIM(CAST(data_nascimento AS VARCHAR)) <> ''
  AND ({dob_valida_sql()}) IS NULL
GROUP BY 1 ORDER BY n DESC LIMIT 20
''').df())

display(con.execute(f'''
SELECT cep, COUNT(*) AS n
FROM {ORIGEM}
WHERE TRIM(CAST(cep AS VARCHAR)) <> ''
  AND ({cep_valido_sql()}) IS NULL
GROUP BY 1 ORDER BY n DESC LIMIT 20
''').df())

,data_nascimento,n
0,2024-09-17,55
1,2024-01-03,54
2,2023-08-16,53
3,2023-01-11,53
4,2023-01-23,52
5,2023-09-05,51
6,2024-06-07,51
7,2024-04-12,51
8,2023-03-28,50
9,2023-06-16,50


,cep,n


In [9]:
# Preenchimento por origem depois da limpeza: é o que o Splink vai ver.
cobertura = ',\n    '.join(
    f'ROUND(100.0 * COUNT({c}) / COUNT(*), 1) AS {c}' for c in texto_cols
)
display(con.execute(f'''
SELECT origem, COUNT(*) AS n_linhas,
    {cobertura}
FROM {TABELA_LIMPA} GROUP BY origem ORDER BY origem
''').df().set_index('origem').T)

origem,censo,cpf
n_linhas,1037775.0,1438943.0
nome_completo,95.6,100.0
primeiro_nome,95.6,100.0
nome_meio,78.2,96.6
ultimo_nome,95.1,100.0
nome_completo_phon,95.6,100.0
primeiro_nome_phon,95.6,100.0
nome_meio_phon,78.2,96.6
ultimo_nome_phon,95.1,100.0
nome_mae,28.9,95.8


## 4. Impacto na coorte

O filtro de óbito não pode comer ground truth. Se um CPF da coorte tem óbito
anterior a 2021 e mesmo assim aparece no Censo 2022, alguma das duas fontes está
errada — e o par sairia da avaliação do NB03 sem aviso.

In [10]:
if not COHORT_DEDUP_ARQUIVO.exists():
    print('Coorte não encontrada, checagem pulada:', COHORT_DEDUP_ARQUIVO)
else:
    con.execute(f'''
    CREATE OR REPLACE TEMP TABLE _cohort_cpf AS
    SELECT DISTINCT {cpf_norm_sql('CPF_NORM')} AS cpf_norm
    FROM read_parquet('{COHORT_DEDUP_ARQUIVO}')
    WHERE CPF_NORM IS NOT NULL
    ''')

    filtro_obito_b = obito_antes_do_censo_sql('b.ano_obito')
    filtro_sem_nome_b = sem_nome_sql('b.nome_completo')
    display(con.execute(f'''
    WITH na_base AS (
        SELECT u.unique_id, u.ano_obito, u.origem, u.nome_completo
        FROM {ORIGEM} u
        JOIN _cohort_cpf k ON u.cpf_norm = k.cpf_norm
    )
    SELECT
        COUNT(*) AS cpf_da_coorte_no_subset,
        SUM(CASE WHEN {filtro_obito_b} THEN 1 ELSE 0 END) AS removidos_por_obito,
        SUM(CASE WHEN NOT {filtro_obito_b} AND {filtro_sem_nome_b}
                 THEN 1 ELSE 0 END) AS removidos_sem_nome,
        SUM(CASE WHEN l.unique_id IS NULL THEN 1 ELSE 0 END) AS removidos_total,
        ROUND(100.0 * SUM(CASE WHEN l.unique_id IS NULL THEN 1 ELSE 0 END)
              / NULLIF(COUNT(*), 0), 3) AS pct
    FROM na_base b
    LEFT JOIN {TABELA_LIMPA} l ON l.unique_id = b.unique_id
    ''').df())

    display(con.execute(f'''
    SELECT u.cpf_norm, u.nome_completo, u.data_nascimento, u.ano_obito
    FROM {ORIGEM} u
    JOIN _cohort_cpf k ON u.cpf_norm = k.cpf_norm
    WHERE {filtro_obito}
    LIMIT 20
    ''').df())


,cpf_da_coorte_no_subset,removidos_por_obito,pct
0,268889,46.0,0.017


,cpf_norm,nome_completo,data_nascimento,ano_obito
0,23898674304,LUIS ANTONIO SILVA RAMOS,1962-09-30,2020
1,00055727387,ANTONIO AROSO MATTOS PEREIRA,1927-06-12,2017
2,28147650353,CLEBER JOAQUIM DUTRA,1967-08-16,2020
3,00176290397,ILAH TORREAO PORTELADA,1925-04-27,2020
4,07484828300,ANTONIO SANTOS RIBEIRO NETO,1950-03-24,1970
5,10322647304,MARIA JOSE PEREIRA ROCHA,1944-04-17,2019
6,00057382379,RAIMUNDA CARDOSO GOMES,1963-09-05,2003
7,00054968372,ANTONIA ARRUDA SOARES,1906-11-08,1999
8,64163725334,ELMICE MARIA AMARAL SA,1963-08-22,1963
9,06718248304,LUIZ BISPO CANTANHEDE,1950-08-19,2017


## 5. Export

In [11]:
print('Exportado:', export_parquet(con, TABELA_LIMPA, path=REGISTRO_LIMPO))
con.close()

Exportado: /home/ibge.gov.br/ramon.goncalves/data/probabilistico_output/registro_limpo.parquet
